In [1]:
import requests
import zipfile
import os
import pandas as pd

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Create the folder to store raw AIS data
os.makedirs("data/raw/ais", exist_ok=True)

# Define what day we want to download (January 15, 2024)
year = "2024"
month = "01"
day = "15"

# Build the download URL from NOAA's bulk data server
url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{year}/AIS_{year}_{month}_{day}.zip"
zip_path = f"data/raw/ais/AIS_{year}_{month}_{day}.zip"
csv_path = f"data/raw/ais/AIS_{year}_{month}_{day}.csv"

print(f"Downloading from: {url}")
print("This may take a few minutes — the file is a few hundred MB...")

# Download the zip file
response = requests.get(url, stream=True)
with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print("Download complete. Unzipping...")

# Unzip it
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("data/raw/ais/")

print(f"Done. CSV ready at: {csv_path}")


This may take a few minutes — the file is a few hundred MB...
Download complete. Unzipping...
Done. CSV ready at: data/raw/ais/AIS_2024_01_15.csv


In [3]:
# Load the CSV into pandas
df = pd.read_csv('data/raw/ais/AIS_2024_01_15.csv')

# First look — how big is this dataset?
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn names:")
print(df.columns.tolist())


Rows: 7,284,415
Columns: 17

Column names:
['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status', 'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass']


In [4]:
# Load the CSV into pandas
df = pd.read_csv('data/raw/ais/AIS_2024_01_15.csv')

# First look — how big is this dataset?
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn names:")
print(df.columns.tolist())

Rows: 7,284,415
Columns: 17

Column names:
['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status', 'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass']


In [5]:
# How does the data actually look? Show first 5 rows
print(df.head())

# What are the data types of each column?
print("\nData types:")
print(df.dtypes)

# How many nulls in each column?
print("\nNull counts per column:")
print(df.isnull().sum())

        MMSI         BaseDateTime       LAT        LON   SOG    COG  Heading  \
0  338467439  2024-01-15T00:00:00  48.75657 -122.50736   0.0  360.0    511.0   
1  367090390  2024-01-15T00:00:00  29.06215  -90.16063   0.3   69.7    511.0   
2  366709780  2024-01-15T00:00:00  47.79480 -122.49430   0.0  298.2    300.0   
3  366943960  2024-01-15T00:00:00  33.35171 -118.26675  26.5  267.0    511.0   
4  338391523  2024-01-15T00:00:01  48.49284 -122.68201   0.0  360.0    511.0   

         VesselName         IMO CallSign  VesselType  Status  Length  Width  \
0              NEVE  IMO0000000      NaN        36.0     NaN     9.0    6.0   
1           SUN DAY         NaN  WDC8625        60.0     0.0    33.0    8.0   
2       WSF SPOKANE  IMO7214325  WYX2004        60.0     0.0   134.0   26.0   
3  CATALINA EXPRESS  IMO8967888  WCJ6210        40.0     0.0    37.0    6.0   
4          RESOLUTE  IMO0000000      NaN        37.0     NaN     9.0    4.0   

   Draft  Cargo TransceiverClass  
0    NaN 

In [6]:
# Check sentinel values across the whole dataset
print("=== SENTINEL VALUE COUNTS ===")
print(f"COG = 360.0 (unavailable): {(df['COG'] == 360.0).sum():,}")
print(f"SOG = 102.3 (unavailable): {(df['SOG'] == 102.3).sum():,}")
print(f"Heading = 511.0 (unavailable): {(df['Heading'] == 511.0).sum():,}")
print(f"IMO = IMO0000000 (fake): {(df['IMO'] == 'IMO0000000').sum():,}")

print("\n=== NULL RATES ===")
for col in df.columns:
    null_count = df[col].isnull().sum()
    pct = (null_count / len(df)) * 100
    if null_count > 0:
        print(f"{col}: {null_count:,} nulls ({pct:.1f}%)")

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicate rows: {df.duplicated().sum():,}")

=== SENTINEL VALUE COUNTS ===
COG = 360.0 (unavailable): 1,163,841
SOG = 102.3 (unavailable): 15,513
Heading = 511.0 (unavailable): 3,724,055
IMO = IMO0000000 (fake): 1,759,288

=== NULL RATES ===
VesselName: 9,356 nulls (0.1%)
IMO: 2,253,683 nulls (30.9%)
CallSign: 792,073 nulls (10.9%)
VesselType: 5,450 nulls (0.1%)
Status: 1,910,957 nulls (26.2%)
Length: 5,458 nulls (0.1%)
Width: 6,254 nulls (0.1%)
Draft: 1,905,557 nulls (26.2%)
Cargo: 1,905,557 nulls (26.2%)

=== DUPLICATE ROWS ===
Duplicate rows: 176


In [7]:
# Write schema notes + hypothesis to docs/
notes = """# AIS Schema Notes — AIS_2024_01_15.csv
Generated from Phase 1 exploration notebook.

## Dataset Overview
- Source: NOAA Marine Cadastre AIS Bulk Data
- Date: January 15, 2024
- Rows: 7,284,415
- Columns: 17

## Column Names
MMSI, BaseDateTime, LAT, LON, SOG, COG, Heading, VesselName, IMO,
CallSign, VesselType, Status, Length, Width, Draft, Cargo, TransceiverClass

## Sentinel Values (look valid but mean unavailable)
- COG = 360.0: 1,163,841 rows (16.0%) — course unavailable
- SOG = 102.3: 15,513 rows (0.2%) — speed unavailable
- Heading = 511.0: 3,724,055 rows (51.1%) — heading unavailable
- IMO = IMO0000000: 1,759,288 rows (24.1%) — fake/missing IMO

## Null Rates
- MMSI: 0% null — most reliable identifier, used as primary key
- IMO: 30.9% null — unreliable as join key
- Status: 26.2% null
- Draft: 26.2% null
- Cargo: 26.2% null
- CallSign: 10.9% null
- VesselName: 0.1% null

## Duplicate Rows
- 176 exact duplicate rows confirmed
- Cause: multiple shore receivers logging the same broadcast

## Key Design Decisions for Phase 2
1. Use MMSI as primary vessel identifier — only field with 0% nulls
2. Flag and exclude sentinel values before any anomaly logic runs
3. Draft and Cargo too sparse for anomaly rules — 26% missing
4. Heading unreliable for direction rules — 51% are sentinel 511.0
5. Deduplicate on (MMSI, BaseDateTime, LAT, LON) before processing

## Project Hypothesis (written before any anomaly rules are built)
Stated on: 2024-01-15 exploration session

We hypothesize that within a single day of AIS data:
- A meaningful number of vessels will exhibit AIS signal gaps
  exceeding a threshold duration (going dark)
- A subset of vessels will show near-zero SOG in open water
  away from known ports for extended periods (loitering)
- A small number of pings will imply physically impossible
  speed between consecutive positions (speed inconsistency)
- Some MMSIs will broadcast conflicting vessel names
  across pings (identity inconsistency)

We further hypothesize that cross-referencing flagged events
with World Port Index proximity data will reduce false positives
by distinguishing legitimate port anchorage from open-water anomalies.

This hypothesis will be compared against actual flag counts,
flag distributions, and false positive rates at project completion.

## What This Hypothesis Cannot Yet Claim
- Weather context not yet confirmed as a viable data source
- Ship-to-ship transfer detection not yet scoped
- Vessel-type-specific thresholds not yet defined
- Multi-day or multi-month patterns not yet in scope
"""

with open('../docs/ais_schema_notes.md', 'w') as f:
    f.write(notes)

print("Schema notes and hypothesis saved to docs/ais_schema_notes.md")

Schema notes and hypothesis saved to docs/ais_schema_notes.md


In [8]:
git add .

SyntaxError: invalid syntax (3827820173.py, line 1)

In [9]:
git add .

SyntaxError: invalid syntax (3827820173.py, line 1)

In [10]:
git status

SyntaxError: invalid syntax (3528599804.py, line 1)

In [11]:
git add docs/phase_plan.md
git commit -m "Add full six-phase project plan to docs"
git push

SyntaxError: invalid syntax (607535023.py, line 1)

In [3]:
# Verify Open-Meteo historical weather API against a real AIS coordinate
import requests
import pandas as pd

# Load AIS data and grab the first row's coordinates and timestamp
df = pd.read_csv('../data/raw/ais/AIS_2024_01_15.csv')
first_row = df.iloc[0]

lat = first_row['LAT']
lon = first_row['LON']
date_str = first_row['BaseDateTime'][:10]  # Extract YYYY-MM-DD from ISO timestamp

print(f"Using coordinate from first AIS row:")
print(f"  MMSI: {first_row['MMSI']}")
print(f"  LAT: {lat}, LON: {lon}")
print(f"  Date: {date_str}")
print()

# Call Open-Meteo historical archive API
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": date_str,
    "end_date": date_str,
    "hourly": "windspeed_10m,visibility",
}

response = requests.get(url, params=params)

print(f"Status code: {response.status_code}")
print(f"URL called: {response.url}")
print()
print("Raw response:")
print(response.json())


Using coordinate from first AIS row:
  MMSI: 338467439
  LAT: 48.75657, LON: -122.50736
  Date: 2024-01-15

Status code: 200
URL called: https://archive-api.open-meteo.com/v1/archive?latitude=48.75657&longitude=-122.50736&start_date=2024-01-15&end_date=2024-01-15&hourly=windspeed_10m%2Cvisibility

Raw response:
{'latitude': 48.822495, 'longitude': -122.49153, 'generationtime_ms': 4.470229148864746, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 0.0, 'hourly_units': {'time': 'iso8601', 'windspeed_10m': 'km/h', 'visibility': 'undefined'}, 'hourly': {'time': ['2024-01-15T00:00', '2024-01-15T01:00', '2024-01-15T02:00', '2024-01-15T03:00', '2024-01-15T04:00', '2024-01-15T05:00', '2024-01-15T06:00', '2024-01-15T07:00', '2024-01-15T08:00', '2024-01-15T09:00', '2024-01-15T10:00', '2024-01-15T11:00', '2024-01-15T12:00', '2024-01-15T13:00', '2024-01-15T14:00', '2024-01-15T15:00', '2024-01-15T16:00', '2024-01-15T17:00', '2024-01-15T18:00', '2024-01-15T19:

In [4]:
# Test visibility on the forecast endpoint — some Open-Meteo variables aren't in the archive API
# Using same lat/lon as previous cell (MMSI 338467439, Puget Sound area)
lat = 48.75657
lon = -122.50736

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "visibility",
    "forecast_days": 1,
}

response = requests.get(url, params=params)
data = response.json()

print(f"Status code: {response.status_code}")
print(f"Visibility unit: {data.get('hourly_units', {}).get('visibility', 'not returned')}")

visibility_values = data.get("hourly", {}).get("visibility", [])
non_null = [v for v in visibility_values if v is not None]

if non_null:
    print(f"Visibility: POPULATED — {len(non_null)}/{len(visibility_values)} hours have values")
    print(f"Sample values (first 6 hours): {visibility_values[:6]}")
else:
    print("Visibility: STILL NULL — not available on forecast endpoint either")


Status code: 200
Visibility unit: m
Visibility: POPULATED — 24/24 hours have values
Sample values (first 6 hours): [19100.0, 18000.0, 19900.0, 18000.0, 17200.0, 16300.0]


In [10]:
## Test for January 15, 2024

lat = 48.75657
lon = -122.50736
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "visibility",
    "start_date": "2024-01-15",
    "end_date": "2024-01-15",
}
response = requests.get(url, params=params)
data = response.json()
print(data)
print(f"Status code: {response.status_code}")
print(f"Time values: {data.get('hourly', {}).get('time', [])[:3]}")
print(f"Visibility values: {data.get('hourly', {}).get('visibility', [])[:6]}")

{'reason': "Parameter 'start_date' is out of allowed range from 2026-03-30 to 2026-07-16", 'error': True}
Status code: 400
Time values: []
Visibility values: []


In [11]:
## Last Check before moving phases 

lat = 48.75657
lon = -122.50736
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "visibility",
    "start_date": "2024-01-15",
    "end_date": "2024-01-15",
}
response = requests.get(url, params=params)
data = response.json()
print(f"Status code: {response.status_code}")
print(f"Response: {data}")

Status code: 200
Response: {'latitude': 48.771976, 'longitude': -122.524734, 'generationtime_ms': 141.34204387664795, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 0.0, 'hourly_units': {'time': 'iso8601', 'visibility': 'm'}, 'hourly': {'time': ['2024-01-15T00:00', '2024-01-15T01:00', '2024-01-15T02:00', '2024-01-15T03:00', '2024-01-15T04:00', '2024-01-15T05:00', '2024-01-15T06:00', '2024-01-15T07:00', '2024-01-15T08:00', '2024-01-15T09:00', '2024-01-15T10:00', '2024-01-15T11:00', '2024-01-15T12:00', '2024-01-15T13:00', '2024-01-15T14:00', '2024-01-15T15:00', '2024-01-15T16:00', '2024-01-15T17:00', '2024-01-15T18:00', '2024-01-15T19:00', '2024-01-15T20:00', '2024-01-15T21:00', '2024-01-15T22:00', '2024-01-15T23:00'], 'visibility': [48200.0, 44400.0, 47800.0, 43400.0, 42100.0, 36900.0, 33700.0, 33100.0, 33900.0, 33300.0, 33000.0, 38300.0, 34400.0, 24000.0, 23600.0, 27000.0, 24400.0, 23900.0, 28600.0, 46900.0, 45200.0, 41300.0, 38700.0, 37400.0]}